### 1. Install Dependencies

In [ ]:
!python -m pip install -q timm pandas matplotlib kaggle scikit-learn wandb seaborn albumentations

### 2. Imports

In [ ]:
import os
import gc
import re
import time
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from tqdm.auto import tqdm
import json
import shutil
import zipfile
import subprocess
from datetime import datetime
import timm
import cv2
import albumentations as A
import numpy as np
import pandas as pd
from PIL import Image
import wandb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.optim.lr_scheduler import _LRScheduler, CosineAnnealingLR

warnings.filterwarnings("ignore")


### 3. Experiment Config & Paths

In [ ]:
# EXPERIMENT ID
EXPERIMENT_CODE = "E01"
SEED = 42

# REPRODUCIBILITY
DETERMINISTIC = True
DETERMINISTIC_WARN_ONLY = True
DISABLE_TF32 = True

# MODEL IDENTIFICATION
MODEL_KIND = "spatial_vit"
IS_VIDEOMAE_NOTEBOOK = False

# FIXED WORKSPACE PATH
WORKSPACE_ROOT = Path("/workspace").resolve()
RUNTIME_ROOT = WORKSPACE_ROOT / "wdf46f_runtime"
DATASET_ROOT = RUNTIME_ROOT / "wilddeepfake-46f"
INDEX_ROOT = Path("/shared-docker")

# Folder output eksperimen.
OUTPUT_ROOT = WORKSPACE_ROOT / "experiments" / MODEL_KIND
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# MODEL PATH
MODEL_NAME = "vit_base_patch16_224.mae"

# Hanya digunakan oleh notebook spatiotemporal VideoMAE.
MODEL_ROOT = RUNTIME_ROOT / "kaggle_models"
MODEL_CHECKPOINT = MODEL_ROOT / "videomae-base-finetuned-ssv2"

# DEVICE DAN AMP
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_ROCM = torch.version.hip is not None
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if IS_ROCM else torch.float16
USE_GRAD_SCALER = USE_AMP and AMP_DTYPE == torch.float16

# HYPERPARAMETER
BATCH_SIZE_CLIPS = 32
EPOCHS = 25
LR = 5e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 8

USE_EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 4

# SCHEDULER
# - "LinearDecayLR"
# - "CosineAnnealingLR"
SCHEDULER_NAME = "LinearDecayLR"

# LinearDecayLR
SCHEDULER_START_DECAY = EPOCHS // 4
SCHEDULER_BOOSTER = 2

# CosineAnnealingLR
SCHEDULER_T_MAX = EPOCHS
SCHEDULER_ETA_MIN = 0.0

# DROPOUT
DROPOUT_RATE = 0.0

# DATASET FILE EXTENSION
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

# Geometry augmentation
AUG_GEOMETRY_ONEOF_PROB = 0.5
AUG_PROB_HFLIP = 0.5

AUG_CROP_LIMIT = 0.15
AUG_CROP_PROB = 0.5

AUG_SCALE_LIMIT = 0.15
AUG_SCALE_PROB = 0.5

# Color / degradation augmentation
AUG_COLORJITTER_PROB = 0.3
AUG_COLORJITTER_BRIGHTNESS = 0.2
AUG_COLORJITTER_CONTRAST = 0.2
AUG_COLORJITTER_SATURATION = 0.2
AUG_COLORJITTER_HUE = 0.2

AUG_BLUR_OR_NOISE_ONEOF_PROB = 0.5

AUG_GAUSSIAN_BLUR_PROB = 0.3
AUG_GAUSSIAN_BLUR_LIMIT = (3, 7)

AUG_GAUSSNOISE_PROB = 0.3
AUG_GAUSSNOISE_VAR_LIMIT = (10.0, 50.0)

# LOSS
USE_FOCAL_LOSS = False
FOCAL_GAMMA = 2.0
USE_FOCAL_ALPHA = True

# NORMALISASI
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)


### 4. Dataset Setup, Kaggle Download, dan Index Validation

In [ ]:
KAGGLE_DATASET_SLUG = "afenmarbun/wilddeepfake-46f"
KAGGLE_MODEL_VERSION_SLUG = "afenmarbun/videomae-base-finetuned-ssv2/pytorch/default/1"
KAGGLE_ZIP_DIR = RUNTIME_ROOT / "kaggle_zip"
FORCE_REDOWNLOAD_DATASET = False
FORCE_REEXTRACT_DATASET = False
FORCE_REDOWNLOAD_MODEL = False
AUTO_DOWNLOAD_DATASET = True
AUTO_DOWNLOAD_MODEL = True

REQUIRED_INDEX_FILES = [
    "dataset_info.json",
    "experiment_configs.json",
    "sampling_preview.csv",
    "train_split.csv",
    "val_split.csv",
    "test_split.csv",
]

REQUIRED_SPLIT_COLUMNS = {
    "label",
    "label_name",
    "clip_dir_rel",
    "num_frames",
    "group_id",
}

REQUIRED_EXPERIMENT_FIELDS = {
    "code",
    "description",
    "num_frames",
    "stride",
    "final_image_size",
    "downscale_first",
}

SPLIT_FILES = {
    "train": "train_split.csv",
    "validation": "val_split.csv",
    "test": "test_split.csv",
}

VALIDATE_ALL_CLIP_PATHS = False
N_CLIP_PATH_SAMPLE = 10

def print_section(title: str):
    print(f"\n{'=' * 12} {title} {'=' * 12}")

def print_kv(key: str, value):
    print(f"{key}: {value}")

def run_command(cmd: list[str]):
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.returncode != 0:
        stdout = result.stdout.strip()
        stderr = result.stderr.strip()
        details = [
            "Perintah gagal dijalankan.",
            f"Return code: {result.returncode}",
            f"Command: {' '.join(cmd)}",
        ]

        if stdout:
            details.append(f"stdout:\n{stdout}")

        if stderr:
            details.append(f"stderr:\n{stderr}")

        raise RuntimeError("\n".join(details))

    return result

def has_directory_content(path: Path, ignored_names: set[str] | None = None) -> bool:
    ignored_names = ignored_names or set()
    if not path.exists():
        return False

    return any(item.name not in ignored_names for item in path.iterdir())

def assert_safe_to_clear(path: Path):
    resolved = path.expanduser().resolve()
    forbidden_paths = {
        Path("/").resolve(),
        Path.home().resolve(),
        WORKSPACE_ROOT.resolve(),
        RUNTIME_ROOT.resolve(),
    }

    if resolved in forbidden_paths or len(resolved.parts) < 3:
        raise RuntimeError(f"Path terlalu berisiko untuk dibersihkan: {resolved}")

def clear_directory(path: Path):
    assert_safe_to_clear(path)
    if not path.exists():
        return

    for item in path.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

def require_existing_dir(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f"{name} tidak ditemukan: {path}")

    if not path.is_dir():
        raise NotADirectoryError(f"{name} bukan folder: {path}")

def require_existing_file(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f"{name} tidak ditemukan: {path}")

    if not path.is_file():
        raise FileNotFoundError(f"{name} bukan file reguler: {path}")

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# KAGGLE CREDENTIAL
def setup_kaggle_credentials():
    print_section("SETUP KAGGLE CREDENTIAL")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json_path = kaggle_dir / "kaggle.json"

    if os.environ.get("KAGGLE_JSON"):
        creds = json.loads(os.environ["KAGGLE_JSON"])
        kaggle_json_path.write_text(json.dumps(creds), encoding="utf-8")
    elif (INDEX_ROOT / "kaggle.json").exists():
        shutil.copy2(INDEX_ROOT / "kaggle.json", kaggle_json_path)
    elif kaggle_json_path.exists():
        pass
    else:
        raise FileNotFoundError(
            "kaggle.json tidak ditemukan.\n"
            "Sediakan credential melalui env KAGGLE_JSON, "
            f"{INDEX_ROOT / 'kaggle.json'}, atau {kaggle_json_path}."
        )

    os.chmod(kaggle_json_path, 0o600)
    print_kv("Kaggle credential", kaggle_json_path)

# DATASET DOWNLOAD AND EXTRACTION
def is_valid_zip(zip_path: Path) -> bool:
    if not zip_path.exists() or zip_path.stat().st_size == 0:
        return False

    try:
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            bad_file = zip_file.testzip()
            if bad_file is not None:
                return False

        return True
    except zipfile.BadZipFile:
        return False

def download_kaggle_dataset() -> list[Path]:
    print_section("DOWNLOAD DATASET DARI KAGGLE")

    KAGGLE_ZIP_DIR.mkdir(parents=True, exist_ok=True)
    zip_files = sorted(KAGGLE_ZIP_DIR.glob("*.zip"))
    valid_zip_files = [zip_file for zip_file in zip_files if is_valid_zip(zip_file)]

    if valid_zip_files and not FORCE_REDOWNLOAD_DATASET:
        return valid_zip_files

    setup_kaggle_credentials()

    for zip_file in zip_files:
        zip_file.unlink()

    run_command([
        "kaggle",
        "datasets",
        "download",
        "-d",
        KAGGLE_DATASET_SLUG,
        "-p",
        str(KAGGLE_ZIP_DIR),
    ])

    zip_files = sorted(KAGGLE_ZIP_DIR.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError("Download selesai, tetapi file ZIP tidak ditemukan.")

    invalid_zip_files = [zip_file for zip_file in zip_files if not is_valid_zip(zip_file)]
    if invalid_zip_files:
        raise RuntimeError(
            "Terdapat ZIP dataset yang tidak valid: "
            f"{[path.name for path in invalid_zip_files]}"
        )

    return zip_files

def extract_dataset(zip_files: list[Path]):
    print_section("EKSTRAKSI DATASET")
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    extract_marker = DATASET_ROOT / ".extract_complete.json"

    dataset_has_content = has_directory_content(
        DATASET_ROOT,
        ignored_names={".extract_complete.json"},
    )

    if extract_marker.exists() and dataset_has_content and not FORCE_REEXTRACT_DATASET:
        return

    if dataset_has_content and not FORCE_REEXTRACT_DATASET:
        print_kv("DATASET_ROOT", DATASET_ROOT)
        return

    if FORCE_REEXTRACT_DATASET:
        clear_directory(DATASET_ROOT)

    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, "r") as zf:
            bad_file = zf.testzip()
            if bad_file is not None:
                raise RuntimeError(f"ZIP tidak valid. File bermasalah: {bad_file}")

            zf.extractall(DATASET_ROOT)

    marker_payload = {
        "dataset_slug": KAGGLE_DATASET_SLUG,
        "dataset_root": str(DATASET_ROOT),
        "zip_files": [str(zip_file) for zip_file in zip_files],
        "completed_at": datetime.utcnow().isoformat() + "Z",
    }

    extract_marker.write_text(
        json.dumps(marker_payload, indent=2),
        encoding="utf-8",
    )

    print_kv("Marker", extract_marker)

def setup_dataset():
    print_section("SETUP DATASET")
    RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    dataset_ready = has_directory_content(
        DATASET_ROOT,
        ignored_names={".extract_complete.json"},
    )

    if dataset_ready and not FORCE_REEXTRACT_DATASET:
        print_kv("DATASET_ROOT", DATASET_ROOT)
        return

    if not AUTO_DOWNLOAD_DATASET:
        raise FileNotFoundError(
            "Dataset belum tersedia dan AUTO_DOWNLOAD_DATASET=False.\n"
            f"DATASET_ROOT: {DATASET_ROOT}"
        )

    zip_files = download_kaggle_dataset()
    extract_dataset(zip_files)

# VIDEOMAE CHECKPOINT DOWNLOAD
def has_hf_checkpoint_files(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False

    has_config = (path / "config.json").exists()
    has_weight = any([
        (path / "model.safetensors").exists(),
        (path / "pytorch_model.bin").exists(),
        (path / "tf_model.h5").exists(),
    ])

    return has_config and has_weight

def setup_videomae_checkpoint():
    print_section("SETUP CHECKPOINT VIDEOMAE")

    if not IS_VIDEOMAE_NOTEBOOK:
        return None

    if has_hf_checkpoint_files(MODEL_CHECKPOINT) and not FORCE_REDOWNLOAD_MODEL:
        print_kv("MODEL_CHECKPOINT", MODEL_CHECKPOINT)
        return MODEL_CHECKPOINT

    if not AUTO_DOWNLOAD_MODEL:
        raise FileNotFoundError(
            "Checkpoint VideoMAE belum tersedia dan AUTO_DOWNLOAD_MODEL=False.\n"
            f"MODEL_CHECKPOINT: {MODEL_CHECKPOINT}"
        )

    setup_kaggle_credentials()

    MODEL_ROOT.mkdir(parents=True, exist_ok=True)

    if FORCE_REDOWNLOAD_MODEL:
        clear_directory(MODEL_ROOT)

    run_command([
        "kaggle",
        "models",
        "instances",
        "versions",
        "download",
        KAGGLE_MODEL_VERSION_SLUG,
        "-p",
        str(MODEL_ROOT),
        "--untar",
    ])

    if not has_hf_checkpoint_files(MODEL_CHECKPOINT):
        raise FileNotFoundError(
            "Download model selesai, tetapi checkpoint tidak ditemukan.\n"
            f"MODEL_CHECKPOINT: {MODEL_CHECKPOINT}"
        )

    return MODEL_CHECKPOINT

# INDEX AND SPLIT VALIDATION
def validate_required_paths():
    print_section("VALIDASI PATH UTAMA")
    require_existing_dir(WORKSPACE_ROOT, "WORKSPACE_ROOT")
    require_existing_dir(DATASET_ROOT, "DATASET_ROOT")
    require_existing_dir(INDEX_ROOT, "INDEX_ROOT")

    for filename in REQUIRED_INDEX_FILES:
        require_existing_file(INDEX_ROOT / filename, filename)

    if IS_VIDEOMAE_NOTEBOOK:
        require_existing_dir(MODEL_CHECKPOINT, "MODEL_CHECKPOINT")
        require_existing_file(MODEL_CHECKPOINT / "config.json", "VideoMAE config.json")

    print_kv("WORKSPACE_ROOT", WORKSPACE_ROOT)
    print_kv("DATASET_ROOT", DATASET_ROOT)
    print_kv("INDEX_ROOT", INDEX_ROOT)

    if IS_VIDEOMAE_NOTEBOOK:
        print_kv("MODEL_CHECKPOINT", MODEL_CHECKPOINT)

def load_index_files():
    print_section("LOAD INDEX DAN SPLIT")
    dataset_info = load_json(INDEX_ROOT / "dataset_info.json")
    experiment_configs = load_json(INDEX_ROOT / "experiment_configs.json")
    sampling_preview_df = pd.read_csv(INDEX_ROOT / "sampling_preview.csv")

    split_dfs = {
        split_name: pd.read_csv(INDEX_ROOT / filename)
        for split_name, filename in SPLIT_FILES.items()
    }

    return dataset_info, experiment_configs, sampling_preview_df, split_dfs

def validate_dataset_info(dataset_info: dict) -> int:
    if "source_clip_len" not in dataset_info:
        raise ValueError("dataset_info.json tidak memiliki field source_clip_len.")

    source_clip_len = int(dataset_info["source_clip_len"])
    if source_clip_len <= 0:
        raise ValueError(f"source_clip_len tidak valid: {source_clip_len}")

    return source_clip_len

def validate_experiment_config(experiment_configs: dict) -> dict:
    if EXPERIMENT_CODE not in experiment_configs:
        raise ValueError(
            f"EXPERIMENT_CODE '{EXPERIMENT_CODE}' tidak ditemukan di experiment_configs.json."
        )

    selected_cfg = experiment_configs[EXPERIMENT_CODE]
    missing_fields = REQUIRED_EXPERIMENT_FIELDS - set(selected_cfg.keys())
    if missing_fields:
        raise ValueError(
            f"Experiment config {EXPERIMENT_CODE} tidak lengkap: {sorted(missing_fields)}"
        )

    return selected_cfg

def validate_split_dfs(split_dfs: dict[str, pd.DataFrame], source_clip_len: int):
    print_section("VALIDASI SPLIT")

    for split_name, df in split_dfs.items():
        if df.empty:
            raise ValueError(f"Split '{split_name}' kosong.")

        missing_columns = REQUIRED_SPLIT_COLUMNS - set(df.columns)
        if missing_columns:
            raise ValueError(
                f"Split '{split_name}' tidak memiliki kolom wajib: {sorted(missing_columns)}. "
                "Kemungkinan file split masih berasal dari Build_Index_and_Splits versi lama."
            )

        invalid_num_frames = df[df["num_frames"].astype(int) != source_clip_len]
        if len(invalid_num_frames) > 0:
            raise ValueError(
                f"Split '{split_name}' memiliki {len(invalid_num_frames)} baris "
                f"dengan num_frames != {source_clip_len}."
            )

    train_groups = set(split_dfs["train"]["group_id"])
    val_groups = set(split_dfs["validation"]["group_id"])
    test_groups = set(split_dfs["test"]["group_id"])

    overlaps = {
        "train-validation": train_groups & val_groups,
        "train-test": train_groups & test_groups,
        "validation-test": val_groups & test_groups,
    }

    leaking_pairs = {
        pair: sorted(list(groups))[:10]
        for pair, groups in overlaps.items()
        if groups
    }
    if leaking_pairs:
        raise ValueError(
            "Data leakage berbasis group_id terdeteksi:\n"
            f"{json.dumps(leaking_pairs, indent=2)}"
        )

def validate_clip_paths(split_dfs: dict[str, pd.DataFrame]):
    print_section("VALIDASI PATH KLIP")
    missing_examples = []
    total_checked = 0

    for split_name, df in split_dfs.items():
        rows = df if VALIDATE_ALL_CLIP_PATHS else df.head(N_CLIP_PATH_SAMPLE)

        for _, row in rows.iterrows():
            clip_path = DATASET_ROOT / str(row["clip_dir_rel"])
            total_checked += 1

            if not clip_path.exists():
                missing_examples.append({
                    "split": split_name,
                    "clip_dir_rel": str(row["clip_dir_rel"]),
                    "expected_path": str(clip_path),
                })

                if len(missing_examples) >= 10:
                    break

        if len(missing_examples) >= 10:
            break

    if missing_examples:
        raise FileNotFoundError(
            "Terdapat path klip yang tidak ditemukan.\n"
            f"{json.dumps(missing_examples, indent=2)}"
        )

    print_kv("Jumlah path dicek", total_checked)

def summarize_dataset(split_dfs: dict[str, pd.DataFrame]):
    print_section("RINGKASAN DATASET")

    for split_name, df in split_dfs.items():
        print_kv(f"{split_name}_num_clips", len(df))
        print_kv(f"{split_name}_num_groups", df["group_id"].nunique())

# EXECUTION
setup_dataset()
setup_videomae_checkpoint()
validate_required_paths()
(
    validated_dataset_info,
    validated_experiment_configs,
    sampling_preview_df,
    validated_split_dfs,
) = load_index_files()
SOURCE_CLIP_LEN = validate_dataset_info(validated_dataset_info)
selected_experiment_config_dict = validate_experiment_config(validated_experiment_configs)
validate_split_dfs(validated_split_dfs, SOURCE_CLIP_LEN)
validate_clip_paths(validated_split_dfs)
summarize_dataset(validated_split_dfs)


### 5. Load Experiment Config

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    code: str
    description: str
    num_frames: int
    stride: int
    final_image_size: int = 224
    downscale_first: int | None = None

CFG = ExperimentConfig(**selected_experiment_config_dict)
OUTPUT_DIR = OUTPUT_ROOT / CFG.code
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 6. W&B Configuration

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "Tugas Akhir"
WANDB_ENTITY = "afenmarbun-institut-teknologi-sumatera"
WANDB_MODE = os.environ.get("WANDB_MODE", "online")
WANDB_GROUP = "Eksperimen Baseline"

def sanitize_wandb_name(text: str) -> str:
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9_.-]+", "_", text)
    return text.strip("_")

WANDB_RUN_NAME = sanitize_wandb_name(
    f"{CFG.code}_{MODEL_KIND}_{MODEL_NAME}"
)

WANDB_TAGS = [
    CFG.code,
    "wilddeepfake",
    "spatial",
    "vit",
    "clip_level",
    "macro_metrics",
]


### 7. Seeding & File Utilities

In [ ]:
def seed_everything(
    seed: int = SEED,
    deterministic: bool = True,
    warn_only: bool = True,
    disable_tf32: bool = True,
):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=warn_only)
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        torch.use_deterministic_algorithms(False)

    if disable_tf32 and torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

        try:
            torch.set_float32_matmul_precision("highest")
        except Exception:
            pass

seed_everything(
    seed=SEED,
    deterministic=DETERMINISTIC,
    warn_only=DETERMINISTIC_WARN_ONLY,
    disable_tf32=DISABLE_TF32,
)

def numeric_key(path: Path):
    nums = re.findall(r"\d+", path.stem)
    return int(nums[-1]) if nums else path.stem

# mengambil daftar file gambar dari sebuah folder, lalu mengurutkannya 
# berdasarkan urutan numerik nama file.
def list_image_files(folder: Path):
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS],
        key=numeric_key,
    )

def ensure_clip_dir(df: pd.DataFrame, dataset_root: Path):
    df = df.copy()

    if "clip_dir_rel" in df.columns:
        def resolve_rel_path(path_value):
            path = Path(str(path_value))
            return str(path if path.is_absolute() else dataset_root / path)

        df["clip_dir"] = df["clip_dir_rel"].apply(resolve_rel_path)
        return df

    if "clip_dir" in df.columns:
        return df

    if "clip_dir_abs" in df.columns:
        df["clip_dir"] = df["clip_dir_abs"].astype(str)
        return df

    raise ValueError(
        "File split tidak memiliki kolom clip_dir_rel, clip_dir, atau clip_dir_abs."
    )

### 8. Prepare Split DataFrames


In [ ]:
train_split_df = ensure_clip_dir(validated_split_dfs["train"], DATASET_ROOT)
val_split_df = ensure_clip_dir(validated_split_dfs["validation"], DATASET_ROOT)
test_df = ensure_clip_dir(validated_split_dfs["test"], DATASET_ROOT)

display(train_split_df.head())
display(sampling_preview_df.head())


### 9. Temporal Sampling Plan

In [ ]:
def compute_temporal_sampling_plan(
    total_frames: int,
    num_frames: int,
    stride: int,
):
    """
    Diketahui:
    - total_frames = N
    - num_frames   = T
    - stride       = tau

    Rumus:
    - L = (T - 1) * tau + 1
    - R = N - L
    - d_kiri  = floor(R / 2)
    - d_kanan = ceil(R / 2)
    """
    if total_frames <= 0:
        raise ValueError("total_frames harus > 0")

    if num_frames <= 0:
        raise ValueError("num_frames harus > 0")

    if stride <= 0:
        raise ValueError("stride harus > 0")

    required_length = (num_frames - 1) * stride + 1
    if required_length > total_frames:
        raise ValueError(
            f"Konfigurasi tidak valid: T={num_frames}, stride={stride}, "
            f"membutuhkan L={required_length}, tetapi klip hanya memiliki N={total_frames}."
        )

    remaining = total_frames - required_length
    discard_left = remaining // 2
    discard_right = remaining - discard_left

    start = discard_left
    end = total_frames - discard_right
    sampled_indices = [start + k * stride for k in range(num_frames)]

    return {
        "N": total_frames,
        "T": num_frames,
        "tau": stride,
        "L": required_length,
        "R": remaining,
        "discard_left": discard_left,
        "discard_right": discard_right,
        "center_start": start,
        "center_end_exclusive": end,
        "sampled_indices": sampled_indices,
    }

def sample_frame_paths_from_clip(
    frame_files: list[Path],
    num_frames: int,
    stride: int,
):
    plan = compute_temporal_sampling_plan(
        total_frames=len(frame_files),
        num_frames=num_frames,
        stride=stride,
    )
    selected = [frame_files[i] for i in plan["sampled_indices"]]
    return selected, plan


### 10. Clip Transform & Augmentation


In [ ]:
def _sample_fakeformer_crop_size(image_size: int, crop_limit: float):
    crop_values = np.arange(0.0, crop_limit, 0.01)

    if len(crop_values) == 0:
        return image_size, image_size

    crop_ratio_h = 1.0 - float(np.random.choice(crop_values))
    crop_ratio_w = 1.0 - float(np.random.choice(crop_values))

    crop_h = max(1, int(crop_ratio_h * image_size))
    crop_w = max(1, int(crop_ratio_w * image_size))

    return crop_h, crop_w

def _make_gauss_noise_transform(p: float):
    try:
        return A.GaussNoise(
            var_limit=AUG_GAUSSNOISE_VAR_LIMIT,
            mean=0,
            per_channel=True,
            p=p,
        )
    except TypeError:
        std_low = math.sqrt(AUG_GAUSSNOISE_VAR_LIMIT[0]) / 255.0
        std_high = math.sqrt(AUG_GAUSSNOISE_VAR_LIMIT[1]) / 255.0

        return A.GaussNoise(
            std_range=(std_low, std_high),
            mean_range=(0.0, 0.0),
            per_channel=True,
            p=p,
        )

class ClipTransform:
    def __init__(
        self,
        image_size: int,
        is_train: bool,
        downscale_first: int | None = None,
        mean: tuple[float, float, float] = NORM_MEAN,
        std: tuple[float, float, float] = NORM_STD,
    ):
        self.image_size = image_size
        self.is_train = is_train
        self.downscale_first = downscale_first
        self.mean = mean
        self.std = std

    def _resize_for_experiment(self, img: Image.Image):
        img = img.convert("RGB")

        if self.downscale_first is not None:
            img = TF.resize(
                img,
                size=[self.downscale_first, self.downscale_first],
                interpolation=InterpolationMode.BILINEAR,
            )
            img = TF.resize(
                img,
                size=[self.image_size, self.image_size],
                interpolation=InterpolationMode.BILINEAR,
            )
        else:
            img = TF.resize(
                img,
                size=[self.image_size, self.image_size],
                interpolation=InterpolationMode.BILINEAR,
            )

        return img

    def _build_train_transform(self):
        crop_h, crop_w = _sample_fakeformer_crop_size(
            image_size=self.image_size,
            crop_limit=AUG_CROP_LIMIT,
        )

        return A.ReplayCompose(
            [
                A.OneOf(
                    [
                        A.RandomCrop(
                            height=crop_h,
                            width=crop_w,
                            p=AUG_CROP_PROB,
                        ),
                        A.RandomScale(
                            scale_limit=AUG_SCALE_LIMIT,
                            interpolation=cv2.INTER_LINEAR,
                            p=AUG_SCALE_PROB,
                        ),
                    ],
                    p=AUG_GEOMETRY_ONEOF_PROB,
                ),
                A.Resize(
                    height=self.image_size,
                    width=self.image_size,
                    interpolation=cv2.INTER_CUBIC,
                    p=1.0,
                ),
                A.HorizontalFlip(
                    p=AUG_PROB_HFLIP,
                ),
                A.ColorJitter(
                    brightness=AUG_COLORJITTER_BRIGHTNESS,
                    contrast=AUG_COLORJITTER_CONTRAST,
                    saturation=AUG_COLORJITTER_SATURATION,
                    hue=AUG_COLORJITTER_HUE,
                    p=AUG_COLORJITTER_PROB,
                ),
                A.OneOf(
                    [
                        A.GaussianBlur(
                            blur_limit=AUG_GAUSSIAN_BLUR_LIMIT,
                            sigma_limit=0,
                            p=AUG_GAUSSIAN_BLUR_PROB,
                        ),
                        _make_gauss_noise_transform(
                            p=AUG_GAUSSNOISE_PROB,
                        ),
                    ],
                    p=AUG_BLUR_OR_NOISE_ONEOF_PROB,
                ),
            ]
        )

    def _normalize_image(self, image) -> torch.Tensor:
        """
        Mengubah input gambar ke format tensor C x H x W,
        melakukan scaling ke rentang [0, 1], clamping,
        dan normalisasi menggunakan mean serta standard deviation.
        """
        tensor = TF.to_image(image)
        tensor = TF.to_dtype(tensor, dtype=torch.float32, scale=True)

        tensor = torch.clamp(tensor, 0.0, 1.0)
        tensor = TF.normalize(
            tensor,
            mean=self.mean,
            std=self.std,
            inplace=True
        )

        return tensor.contiguous()

    def __call__(self, pil_images: list[Image.Image]):
        if len(pil_images) == 0:
            raise ValueError("pil_images kosong. Dataset menghasilkan klip tanpa frame.")

        resized_images = [
            self._resize_for_experiment(img)
            for img in pil_images
        ]

        if self.is_train:
            transform = self._build_train_transform()
            first_image = np.array(resized_images[0])
            first_aug = transform(image=first_image)
            replay = first_aug["replay"]
            augmented_arrays = [first_aug["image"]]

            for img in resized_images[1:]:
                image_array = np.array(img)
                replayed = A.ReplayCompose.replay(
                    replay,
                    image=image_array,
                )
                augmented_arrays.append(replayed["image"])

        else:
            augmented_arrays = [
                np.array(img)
                for img in resized_images
            ]

        tensors = [
            self._normalize_image(img)
            for img in augmented_arrays
        ]

        return torch.stack(tensors, dim=0)  # [T, C, H, W]

### 11. Dataset Class


In [ ]:
class SpatialClipDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        experiment_cfg: ExperimentConfig,
        is_train: bool,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.cfg = experiment_cfg
        self.is_train = is_train
        self.transform = ClipTransform(
            image_size=self.cfg.final_image_size,
            is_train=is_train,
            downscale_first=self.cfg.downscale_first,
            mean=NORM_MEAN,
            std=NORM_STD,
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip_dir = Path(row["clip_dir"])
        label = int(row["label"])
        frame_files = list_image_files(clip_dir)

        if len(frame_files) != SOURCE_CLIP_LEN:
            raise ValueError(
                f"Folder {clip_dir} berisi {len(frame_files)} frame, "
                f"padahal harus {SOURCE_CLIP_LEN} frame."
            )

        selected_files, plan = sample_frame_paths_from_clip(
            frame_files=frame_files,
            num_frames=self.cfg.num_frames,
            stride=self.cfg.stride,
        )

        images = []
        for fp in selected_files:
            with Image.open(fp) as img:
                images.append(img.convert("RGB"))

        clip_tensor = self.transform(images)
        label_tensor = torch.tensor(label, dtype=torch.long)

        meta = {
            "clip_dir": str(clip_dir),
            "label_name": row["label_name"],
            "sampled_indices": torch.tensor(plan["sampled_indices"], dtype=torch.long),
            "group_id": row["group_id"] if "group_id" in row.index else None,
            "source_clip_name": row["source_clip_name"] if "source_clip_name" in row.index else None,
            "clip_folder_name": row["clip_folder_name"] if "clip_folder_name" in row.index else None,
        }

        return clip_tensor, label_tensor, meta


### 12. DataLoaders


In [ ]:
train_dataset = SpatialClipDataset(
    dataframe=train_split_df,
    experiment_cfg=CFG,
    is_train=True,
)

val_dataset = SpatialClipDataset(
    dataframe=val_split_df,
    experiment_cfg=CFG,
    is_train=False,
)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

def make_loader(dataset, shuffle: bool, seed_offset: int = 0):
    generator = torch.Generator()
    generator.manual_seed(SEED + seed_offset)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE_CLIPS,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
        worker_init_fn=seed_worker,
        generator=generator,
    )

train_loader = make_loader(train_dataset, shuffle=True, seed_offset=11)
val_loader = make_loader(val_dataset, shuffle=False, seed_offset=22)


### 13. Focal Loss


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha=None, reduction: str = "mean"):
        super().__init__()

        if gamma < 0:
            raise ValueError("gamma harus bernilai >= 0.")

        if reduction not in {"mean", "sum", "none"}:
            raise ValueError("reduction harus salah satu dari: 'mean', 'sum', atau 'none'.")

        self.gamma = gamma
        self.reduction = reduction

        if alpha is None:
            self.register_buffer("alpha", None)
        else:
            alpha = torch.as_tensor(alpha, dtype=torch.float32)
            self.register_buffer("alpha", alpha)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.long()

        log_probs = torch.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)

        target_indices = targets.unsqueeze(1)
        log_pt = log_probs.gather(1, target_indices).squeeze(1)
        pt = probs.gather(1, target_indices).squeeze(1)

        focal_factor = (1.0 - pt).pow(self.gamma)
        loss = -focal_factor * log_pt

        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            loss = alpha_t * loss

        if self.reduction == "mean":
            return loss.mean()

        if self.reduction == "sum":
            return loss.sum()

        return loss


### 14. Scheduler


In [ ]:
class LinearDecayLR(_LRScheduler):
    def __init__(self, optimizer, n_epoch, start_decay, last_epoch=-1, booster=2):
        if n_epoch <= 0:
            raise ValueError("n_epoch harus > 0")

        if start_decay < 0 or start_decay >= n_epoch:
            raise ValueError("start_decay harus berada pada rentang [0, n_epoch)")

        self.start_decay = start_decay
        self.n_epoch = n_epoch
        self.booster = booster
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        last_epoch = self.last_epoch
        base_lr = self.base_lrs[-1]
        boosted_lr = base_lr * self.booster

        if last_epoch < self.start_decay:
            if self.start_decay == 0:
                lr = boosted_lr
            else:
                lr = base_lr + (boosted_lr - base_lr) / self.start_decay * last_epoch
        else:
            decay_span = self.n_epoch - self.start_decay
            lr = boosted_lr - (boosted_lr / decay_span) * (last_epoch - self.start_decay)

        self._last_lr = [lr]
        return [lr for _ in self.optimizer.param_groups]

def build_scheduler(optimizer):
    if SCHEDULER_NAME == "LinearDecayLR":
        return LinearDecayLR(
            optimizer=optimizer,
            n_epoch=EPOCHS,
            start_decay=SCHEDULER_START_DECAY,
            booster=SCHEDULER_BOOSTER,
        )

    if SCHEDULER_NAME == "CosineAnnealingLR":
        return CosineAnnealingLR(
            optimizer=optimizer,
            T_max=SCHEDULER_T_MAX,
            eta_min=SCHEDULER_ETA_MIN,
        )

    raise ValueError(f"SCHEDULER_NAME tidak dikenali: {SCHEDULER_NAME}")


### 15. Model, Loss, Optimizer, Scheduler


In [ ]:
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=2,
    img_size=CFG.final_image_size,
    drop_rate=DROPOUT_RATE,
)

model = model.to(DEVICE)

if USE_FOCAL_LOSS:
    if USE_FOCAL_ALPHA:
        label_counts_series = train_split_df["label"].astype(int).value_counts().sort_index()
        class_counts = torch.tensor(
            [
                label_counts_series.get(0, 0),
                label_counts_series.get(1, 0),
            ],
            dtype=torch.float32,
        )

        if torch.any(class_counts == 0):
            raise ValueError(
                f"Terdapat kelas dengan jumlah sampel 0 pada data latih: {class_counts.tolist()}"
            )

        focal_alpha = class_counts.sum() / (len(class_counts) * class_counts)
        focal_alpha = focal_alpha / focal_alpha.mean()
    else:
        focal_alpha = None

    criterion = FocalLoss(
        gamma=FOCAL_GAMMA,
        alpha=focal_alpha,
        reduction="mean",
    ).to(DEVICE)
else:
    criterion = nn.CrossEntropyLoss().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = build_scheduler(optimizer)
scaler = torch.cuda.amp.GradScaler(enabled=USE_GRAD_SCALER)

### 16. Metrics


In [ ]:
def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "precision_weighted": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "cm": cm,
        "y_pred": y_pred,
        "classification_report_text": classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["real", "fake"],
            digits=4,
            zero_division=0,
        ),
        "classification_report_dict": classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["real", "fake"],
            digits=4,
            zero_division=0,
            output_dict=True,
        ),
    }

    return metrics

### 17. Train/Eval Functions


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
    epoch: int | None = None,
    total_epochs: int | None = None,
):
    model.train()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []

    desc = f"Train {epoch:02d}/{total_epochs}" if epoch is not None and total_epochs is not None else "Train"
    progress_bar = tqdm(
        loader,
        desc=desc,
        total=len(loader),
        dynamic_ncols=True,
        leave=False,
    )

    for clips, labels, _ in progress_bar:
        batch_start = time.time()
        batch_size, num_frames, num_channels, height, width = clips.shape

        clips = clips.to(device, non_blocking=True) # [B, T, C, H, W]
        labels = labels.to(device, non_blocking=True)
        frames = clips.reshape(batch_size * num_frames, num_channels, height, width)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            frame_logits = model(frames)
            clip_logits = frame_logits.view(batch_size, num_frames, 2).mean(dim=1)
            loss = criterion(clip_logits, labels)
            preds = torch.argmax(clip_logits.detach().float(), dim=1)
            all_y_true.extend(labels.detach().cpu().numpy().tolist())
            all_y_pred.extend(preds.detach().cpu().numpy().tolist())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)

        avg_loss = running_loss / max(total_samples, 1)
        batch_time = time.time() - batch_start
        progress_bar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "batch_s": f"{batch_time:.1f}",
        })

        del clips, labels, frames, frame_logits, clip_logits, loss

    train_metrics = compute_classification_metrics(all_y_true, all_y_pred)
    train_metrics["loss"] = running_loss / max(total_samples, 1)
    return train_metrics

@torch.no_grad()
def evaluate_clip_level(
    model,
    loader,
    criterion,
    device,
    epoch: int | None = None,
    total_epochs: int | None = None,
    split_name: str = "Valid",
):
    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []
    all_probs = []
    all_clip_dirs = []

    desc = f"{split_name} {epoch:02d}/{total_epochs}" if epoch is not None and total_epochs is not None else split_name
    progress_bar = tqdm(
        loader,
        desc=desc,
        total=len(loader),
        dynamic_ncols=True,
        leave=False,
    )

    for clips, labels, meta in progress_bar:
        batch_start = time.time()
        batch_size, num_frames, num_channels, height, width = clips.shape

        clips = clips.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        frames = clips.reshape(batch_size * num_frames, num_channels, height, width)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            frame_logits = model(frames)
            clip_logits = frame_logits.view(batch_size, num_frames, 2).mean(dim=1)
            loss = criterion(clip_logits, labels)

        clip_probs = torch.softmax(clip_logits.float(), dim=1)
        preds = torch.argmax(clip_probs, dim=1)
        
        # dipindahkan ke cpu karena
        # Tensor yang masih berada di GPU tidak bisa langsung dikonversi ke NumPy.
        y_true = labels.detach().cpu().numpy()
        y_pred = preds.detach().cpu().numpy()
        prob_np = clip_probs.detach().cpu().numpy()

        all_y_true.extend(y_true.tolist())
        all_y_pred.extend(y_pred.tolist())
        all_probs.append(prob_np)
        all_clip_dirs.extend(meta["clip_dir"])

        running_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)

        avg_loss = running_loss / max(total_samples, 1)
        batch_time = time.time() - batch_start
        progress_bar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "batch_s": f"{batch_time:.1f}",
        })

        del clips, labels, frames, frame_logits, clip_logits, clip_probs, loss

    all_probs = np.concatenate(all_probs, axis=0)

    metric_dict = compute_classification_metrics(all_y_true, all_y_pred)
    metric_dict["loss"] = running_loss / max(total_samples, 1)
    metric_dict["y_true"] = np.array(all_y_true)
    metric_dict["y_pred"] = np.array(all_y_pred)
    metric_dict["y_prob_real"] = all_probs[:, 0]
    metric_dict["y_prob_fake"] = all_probs[:, 1]
    metric_dict["clip_dirs"] = list(all_clip_dirs)
    return metric_dict


### 18. W&B Init


In [ ]:
wandb_run = None

def setup_wandb_login():
    if not USE_WANDB:
        return

    os.environ["WANDB_MODE"] = WANDB_MODE

def build_wandb_config():
    config = {
        "experiment_code": CFG.code,
        "experiment_description": CFG.description,
        "model_kind": MODEL_KIND,
        "model_name": MODEL_NAME,
        "num_classes": 2,
        "class_0": "real",
        "class_1": "fake",
        "source_clip_len": SOURCE_CLIP_LEN,
        "num_frames": CFG.num_frames,
        "stride": CFG.stride,
        "final_image_size": CFG.final_image_size,
        "downscale_first": CFG.downscale_first,
        "aug_geometry_oneof_prob": AUG_GEOMETRY_ONEOF_PROB,
        "aug_horizontal_flip": AUG_PROB_HFLIP,
        "aug_crop_limit": AUG_CROP_LIMIT,
        "aug_crop_prob": AUG_CROP_PROB,
        "aug_scale_limit": AUG_SCALE_LIMIT,
        "aug_scale_prob": AUG_SCALE_PROB,
        "aug_colorjitter_prob": AUG_COLORJITTER_PROB,
        "aug_colorjitter_brightness": AUG_COLORJITTER_BRIGHTNESS,
        "aug_colorjitter_contrast": AUG_COLORJITTER_CONTRAST,
        "aug_colorjitter_saturation": AUG_COLORJITTER_SATURATION,
        "aug_colorjitter_hue": AUG_COLORJITTER_HUE,
        "aug_blur_or_noise_oneof_prob": AUG_BLUR_OR_NOISE_ONEOF_PROB,
        "aug_gaussian_blur_prob": AUG_GAUSSIAN_BLUR_PROB,
        "aug_gaussian_blur_limit": AUG_GAUSSIAN_BLUR_LIMIT,
        "aug_gaussnoise_prob": AUG_GAUSSNOISE_PROB,
        "aug_gaussnoise_var_limit": AUG_GAUSSNOISE_VAR_LIMIT,
        "batch_size_clips": BATCH_SIZE_CLIPS,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout_rate": DROPOUT_RATE,
        "optimizer": optimizer.__class__.__name__,
        "scheduler": SCHEDULER_NAME,
        "loss_name": criterion.__class__.__name__,
        "use_focal_loss": USE_FOCAL_LOSS,
        "focal_gamma": FOCAL_GAMMA if USE_FOCAL_LOSS else None,
        "use_focal_alpha": USE_FOCAL_ALPHA if USE_FOCAL_LOSS else None,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "normalization_mean": NORM_MEAN,
        "normalization_std": NORM_STD,
        "checkpoint_selected_by": "val_accuracy",
        "evaluation_level": "clip_level",
    }

    if SCHEDULER_NAME == "LinearDecayLR":
        config.update({
            "scheduler_start_decay": SCHEDULER_START_DECAY,
            "scheduler_booster": SCHEDULER_BOOSTER,
            "base_learning_rate": LR,
            "max_active_learning_rate": LR * SCHEDULER_BOOSTER,
        })
    elif SCHEDULER_NAME == "CosineAnnealingLR":
        config.update({
            "scheduler_t_max": SCHEDULER_T_MAX,
            "scheduler_eta_min": SCHEDULER_ETA_MIN,
        })

    if USE_EARLY_STOPPING:
        config["early_stopping_patience"] = EARLY_STOPPING_PATIENCE

    return config

if USE_WANDB:
    setup_wandb_login()

    if wandb.run is not None:
        wandb.finish()

    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        tags=WANDB_TAGS,
        config=build_wandb_config(),
        dir=str(OUTPUT_DIR),
        reinit=True,
    )

    wandb_run.define_metric("epoch")
    wandb_run.define_metric("train/*", step_metric="epoch")
    wandb_run.define_metric("valid/*", step_metric="epoch")
    wandb_run.define_metric("lr", step_metric="epoch")
    wandb_run.define_metric("time/*", step_metric="epoch")
    wandb_run.define_metric("test/*")


### 19. Training Loop & Checkpoint


In [ ]:
history = []
best_state = None
best_val_accuracy = float("-inf")
patience_counter = 0
checkpoint_path = OUTPUT_DIR / f"best_{CFG.code}_spatial_vit.pt"
training_time_sec = 0.0
start_time = time.time()

epoch_bar = tqdm(
    range(1, EPOCHS + 1),
    desc=f"Training {CFG.code}",
    total=EPOCHS,
    dynamic_ncols=True,
)

for epoch in epoch_bar:
    epoch_start = time.time()

    train_result = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=DEVICE,
        epoch=epoch,
        total_epochs=EPOCHS,
    )
    train_loss = train_result["loss"]

    val_result = evaluate_clip_level(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        epoch=epoch,
        total_epochs=EPOCHS,
        split_name="Valid",
    )

    scheduler.step()
    epoch_time_sec = time.time() - epoch_start

    row = {
        "epoch": int(epoch),
        "train_loss": float(train_loss),
        "train_accuracy": float(train_result["accuracy"]),
        "train_precision_macro": float(train_result["precision_macro"]),
        "train_recall_macro": float(train_result["recall_macro"]),
        "train_f1_macro": float(train_result["f1_macro"]),
        "train_precision_weighted": float(train_result["precision_weighted"]),
        "train_recall_weighted": float(train_result["recall_weighted"]),
        "train_f1_weighted": float(train_result["f1_weighted"]),
        "val_loss": float(val_result["loss"]),
        "val_accuracy": float(val_result["accuracy"]),
        "val_precision_macro": float(val_result["precision_macro"]),
        "val_recall_macro": float(val_result["recall_macro"]),
        "val_f1_macro": float(val_result["f1_macro"]),
        "val_precision_weighted": float(val_result["precision_weighted"]),
        "val_recall_weighted": float(val_result["recall_weighted"]),
        "val_f1_weighted": float(val_result["f1_weighted"]),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "epoch_time_sec": float(epoch_time_sec),
    }
    history.append(row)

    if wandb_run is not None:
        wandb_run.log(
            {
                "epoch": row["epoch"],
                "train/loss": row["train_loss"],
                "train/accuracy": row["train_accuracy"],
                "train/precision_macro": row["train_precision_macro"],
                "train/recall_macro": row["train_recall_macro"],
                "train/f1_macro": row["train_f1_macro"],
                "train/precision_weighted": row["train_precision_weighted"],
                "train/recall_weighted": row["train_recall_weighted"],
                "train/f1_weighted": row["train_f1_weighted"],
                "valid/loss": row["val_loss"],
                "valid/accuracy": row["val_accuracy"],
                "valid/precision_macro": row["val_precision_macro"],
                "valid/recall_macro": row["val_recall_macro"],
                "valid/f1_macro": row["val_f1_macro"],
                "valid/precision_weighted": row["val_precision_weighted"],
                "valid/recall_weighted": row["val_recall_weighted"],
                "valid/f1_weighted": row["val_f1_weighted"],
                "lr": row["lr"],
                "time/epoch_sec": row["epoch_time_sec"],
            },
            step=epoch,
        )

    epoch_bar.set_postfix({
        "train_loss": f"{train_loss:.4f}",
        "val_loss": f"{val_result['loss']:.4f}",
        "val_f1_macro": f"{val_result['f1_macro']:.4f}",
    })

    current_val_accuracy = float(val_result["accuracy"])
    if current_val_accuracy > best_val_accuracy:
        best_val_accuracy = current_val_accuracy
        patience_counter = 0

        model_state_cpu = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        best_state = {
            "epoch": int(epoch),
            "experiment_code": str(CFG.code),
            "experiment_config": dict(CFG.__dict__),
            "model_name": str(MODEL_NAME),
            "scheduler_name": SCHEDULER_NAME,
            "checkpoint_selected_by": "val_accuracy",
            "loss_name": str(criterion.__class__.__name__),
            "use_focal_loss": bool(USE_FOCAL_LOSS),
            "focal_gamma": float(FOCAL_GAMMA) if USE_FOCAL_LOSS else None,
            "use_focal_alpha": bool(USE_FOCAL_ALPHA) if USE_FOCAL_LOSS else None,
            "model_state_dict": model_state_cpu,
            "best_val_accuracy": float(val_result["accuracy"]),
            "best_val_loss": float(val_result["loss"]),
            "best_val_precision_macro": float(val_result["precision_macro"]),
            "best_val_recall_macro": float(val_result["recall_macro"]),
            "best_val_f1_macro": float(val_result["f1_macro"]),
            "best_val_precision_weighted": float(val_result["precision_weighted"]),
            "best_val_recall_weighted": float(val_result["recall_weighted"]),
            "best_val_f1_weighted": float(val_result["f1_weighted"]),
        }

        torch.save(best_state, checkpoint_path)

        if wandb_run is not None:
            wandb_run.summary["best_epoch"] = int(epoch)
            wandb_run.summary["best/val_accuracy"] = float(val_result["accuracy"])
            wandb_run.summary["best/val_loss"] = float(val_result["loss"])
            wandb_run.summary["best/val_precision_macro"] = float(val_result["precision_macro"])
            wandb_run.summary["best/val_recall_macro"] = float(val_result["recall_macro"])
            wandb_run.summary["best/val_f1_macro"] = float(val_result["f1_macro"])
            wandb_run.summary["best/val_precision_weighted"] = float(val_result["precision_weighted"])
            wandb_run.summary["best/val_recall_weighted"] = float(val_result["recall_weighted"])
            wandb_run.summary["best/val_f1_weighted"] = float(val_result["f1_weighted"])
    elif USE_EARLY_STOPPING:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            break

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

training_time_sec = time.time() - start_time
history_df = pd.DataFrame(history)
history_df = history_df[
    [
        "epoch",
        "train_loss",
        "train_accuracy",
        "train_precision_macro",
        "train_recall_macro",
        "train_f1_macro",
        "train_precision_weighted",
        "train_recall_weighted",
        "train_f1_weighted",
        "val_loss",
        "val_accuracy",
        "val_precision_macro",
        "val_recall_macro",
        "val_f1_macro",
        "val_precision_weighted",
        "val_recall_weighted",
        "val_f1_weighted",
        "lr",
        "epoch_time_sec",
    ]
]

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

if wandb_run is not None:
    wandb_run.summary["training_time_sec"] = float(training_time_sec)
    wandb_run.log(
        {
            "training/history_table": wandb.Table(dataframe=history_df)
        },
        step=int(history_df["epoch"].max()),
    )

display(history_df.tail().reset_index(drop=True))


### 20. Training Summary


In [ ]:
if best_state is None:
    raise RuntimeError(
        "best_state masih None. Artinya belum ada checkpoint yang berhasil disimpan."
    )

summary_df = pd.DataFrame([{
    "experiment_code": CFG.code,
    "experiment_description": CFG.description,
    "model_kind": MODEL_KIND,
    "model_name": MODEL_NAME,
    "checkpoint_path": str(checkpoint_path),
    "num_frames": CFG.num_frames,
    "stride": CFG.stride,
    "final_image_size": CFG.final_image_size,
    "downscale_first": CFG.downscale_first,
    "batch_size_clips": BATCH_SIZE_CLIPS,
    "scheduler": SCHEDULER_NAME,
    "training_time_sec": float(training_time_sec),
    "checkpoint_selected_by": best_state.get("checkpoint_selected_by", "val_accuracy"),
    "loss_name": best_state.get("loss_name", criterion.__class__.__name__),
    "use_focal_loss": best_state.get("use_focal_loss", USE_FOCAL_LOSS),
    "focal_gamma": best_state.get("focal_gamma", FOCAL_GAMMA if USE_FOCAL_LOSS else None),
    "use_focal_alpha": best_state.get("use_focal_alpha", USE_FOCAL_ALPHA if USE_FOCAL_LOSS else None),
    "best_epoch": best_state["epoch"],
    "best_val_accuracy": best_state["best_val_accuracy"],
    "best_val_loss": best_state["best_val_loss"],
    "best_val_precision_macro": best_state["best_val_precision_macro"],
    "best_val_recall_macro": best_state["best_val_recall_macro"],
    "best_val_f1_macro": best_state["best_val_f1_macro"],
    "best_val_precision_weighted": best_state["best_val_precision_weighted"],
    "best_val_recall_weighted": best_state["best_val_recall_weighted"],
    "best_val_f1_weighted": best_state["best_val_f1_weighted"],
}])

summary_df.to_csv(OUTPUT_DIR / "training_summary.csv", index=False)

if wandb_run is not None:
    wandb_run.log({
        "training/summary_table": wandb.Table(dataframe=summary_df)
    })

display(summary_df)


### 21. Log W&B Artifact


In [ ]:
def log_experiment_artifact_to_wandb():
    if wandb_run is None:
        return

    artifact_name = sanitize_wandb_name(f"{WANDB_RUN_NAME}_outputs")

    artifact = wandb.Artifact(
        name=artifact_name,
        type="experiment-output",
        metadata={
            "experiment_code": CFG.code,
            "model_kind": MODEL_KIND,
            "num_frames": CFG.num_frames,
            "stride": CFG.stride,
            "final_image_size": CFG.final_image_size,
            "checkpoint_selected_by": "val_accuracy",
        },
    )

    candidate_files = [
        checkpoint_path,
        OUTPUT_DIR / "training_history.csv",
        OUTPUT_DIR / "training_summary.csv",
    ]

    for file_path in candidate_files:
        file_path = Path(file_path)
        if file_path.exists():
            artifact.add_file(str(file_path))

    wandb_run.log_artifact(artifact)

log_experiment_artifact_to_wandb()

if wandb_run is not None:
    wandb_run.finish()